# LAB - Cruce ITT x IPM (Ciudad Paraiso)

Mapa de **laboratorio** para evaluar la integracion del Indice de Pobreza
Multidimensional (IPM) con el ITT. No sustituye a los mapas de produccion; sirve para
probar la experiencia antes de llevarla a los demas.

Panel analitico reactivo:
- **Matriz de priorizacion (4 cuadrantes):** cada manzana es un punto. Eje X = IPM
  (vulnerabilidad), Eje Y = eventos de seguridad por manzana (presion). Lineas = promedio
  de la zona. Color por cuadrante (Okabe-Ito).
- **Barras comparativas:** al seleccionar una manzana, IPM e ITT de esa manzana vs el
  promedio de la zona (0-100), mas sus principales privaciones.
- **Metrica contextual por capa:** al encender una capa de eventos (homicidios/hurtos),
  informa cuantos eventos caen en manzanas de IPM alto.

Nota: el ITT por manzana hoy es uniforme (58.5); por eso el eje Y de la matriz usa los
eventos de seguridad reales por manzana (que si varian). El ITT se muestra como referencia
en las barras.


## 0. Setup Colab (opcional)

In [ ]:
# === SETUP COLAB (se ignora en local) ===
def _en_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False
if _en_colab():
    import os, subprocess, sys, zipfile
    REPO = "https://github.com/j0rg3c45/mapa_Analitico_ciudad_Paraiso.git"
    DEST = "/content/mapa_Analitico_ciudad_Paraiso"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "geopandas", "folium", "mapclassify", "pyarrow", "openpyxl"], check=False)
    if not os.path.isdir(DEST):
        subprocess.run(["git", "clone", "--depth", "1", REPO, DEST], check=False)
    _zip = os.path.join(DEST, "data_publico", "data_itt_publico.zip")
    _data_dir = os.path.join(DEST, "data")
    if os.path.isfile(_zip):
        os.makedirs(_data_dir, exist_ok=True)
        with zipfile.ZipFile(_zip) as z:
            z.extractall(_data_dir)
        print("Datos publicos descomprimidos en:", os.path.join(_data_dir, "1_data_itt"))
    else:
        print("Aviso: no se encontro", _zip, "- revisa que el repo lo incluya.")
    _nb_dir = os.path.join(DEST, "notebooks")
    if os.path.isdir(_nb_dir):
        os.chdir(_nb_dir)
    print("Entorno Colab listo. Directorio de trabajo:", os.getcwd())
else:
    print("Entorno local (uv). Setup Colab omitido.")

## 1. Entorno y paleta Okabe-Ito

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import geopandas as gpd
OKI_AZUL="#0072B2"; OKI_NARANJA="#E69F00"; OKI_VERDE="#009E73"
OKI_AMARILLO="#F0E442"; OKI_AZUL_C="#56B4E9"; OKI_BERMELL="#D55E00"
OKI_MORADO="#CC79A7"; OKI_GRIS="#333333"
BG="#F4F6F9"
CRS_WEB=4326; CRS_METRIC=9377
print("Entorno listo.")

## 2. Rutas y poligono de la zona

In [ ]:
def _resolver_poligono():
    env=os.environ.get("POLIGONO_ZONA"); cand=[env] if env else []
    cand+=[os.path.join("..","data","geojson","poligono_ciudad_Paraiso.geojson"),
           os.path.join("data","geojson","poligono_ciudad_Paraiso.geojson"),
           "/content/data/geojson/poligono_ciudad_Paraiso.geojson",
           "/content/drive/MyDrive/mapa_Analitico_ciudad_Paraiso/data/geojson/poligono_ciudad_Paraiso.geojson"]
    for p in cand:
        if p and os.path.isfile(p): return os.path.abspath(p)
    return None
_pp=_resolver_poligono()
POLIGONO_ZONA=gpd.read_file(_pp).to_crs(CRS_WEB) if _pp else None
print("Poligono:", _pp)
OUT_DIR=None
for _b in [os.path.join("..","data_processed_geo"),"data_processed_geo","/content/data_processed_geo"]:
    try:
        os.makedirs(_b,exist_ok=True); OUT_DIR=os.path.abspath(_b); break
    except Exception: continue
MAPAS_DIR=os.path.abspath(os.path.join(os.path.dirname(OUT_DIR.rstrip(os.sep)),"mapa_html"))
os.makedirs(MAPAS_DIR,exist_ok=True)
print("Salida de mapas HTML:", MAPAS_DIR)
def _ruta_itt(n):
    for p in [os.path.join("..","data","1_data_itt",n),os.path.join("data","1_data_itt",n),
              os.path.join("/content","data","1_data_itt",n),
              os.path.join("/content","drive","MyDrive","mapa_Analitico_ciudad_Paraiso","data","1_data_itt",n)]:
        if os.path.isfile(p): return os.path.abspath(p)
    return None
def _ruta_geojson(n):
    for p in [os.path.join("..","data","geojson",n),os.path.join("data","geojson",n),
              os.path.join("/content","data","geojson",n),
              os.path.join("/content","drive","MyDrive","mapa_Analitico_ciudad_Paraiso","data","geojson",n)]:
        if os.path.isfile(p): return os.path.abspath(p)
    return None

## 3. Carga y cruce de datos (IPM + ITT + eventos por manzana)

In [ ]:
# 3a. Carga de IPM (recortado a la zona) + cruce con eventos de seguridad por manzana.
_rp_ipm = _ruta_geojson("Mzn_ipm_variables.geojson")
if _rp_ipm is None:
    raise FileNotFoundError("No se encontro Mzn_ipm_variables.geojson en data/geojson.")
_raw = gpd.read_file(_rp_ipm)
_raw = _raw.set_crs(epsg=CRS_WEB) if _raw.crs is None else _raw.to_crs(CRS_WEB)
_raw = _raw[_raw.geometry.notna()].copy()
_raw["ipm"] = pd.to_numeric(_raw["ipm"], errors="coerce")
if POLIGONO_ZONA is not None:
    _zona = POLIGONO_ZONA.geometry.union_all()
    ipm_z = _raw[_raw.geometry.within(_zona) | _raw.geometry.intersects(_zona)].copy()
else:
    ipm_z = _raw.copy()
ipm_z = ipm_z.reset_index(drop=True)
ipm_z["mz_id"] = range(len(ipm_z))
# ITT por manzana (consolidado): hoy uniforme (58.5). Se cruza por join espacial
# (centroide de la manzana IPM dentro de la manzana ITT) para traer itt y nivel.
_rp_mz = _ruta_itt("01_manzanas_itt_consolidado.geojson")
ITT_ZONA = None
if _rp_mz is not None:
    _mz = gpd.read_file(_rp_mz).to_crs(CRS_WEB)
    _cent = ipm_z.copy()
    _cent["geometry"] = _cent.geometry.representative_point()
    _j = gpd.sjoin(_cent[["mz_id", "geometry"]], _mz[["itt", "nivel", "geometry"]],
                   how="left", predicate="within")
    _j = _j.drop_duplicates("mz_id").set_index("mz_id")
    ipm_z["itt"] = ipm_z["mz_id"].map(_j["itt"])
    ipm_z["nivel"] = ipm_z["mz_id"].map(_j["nivel"])
    ITT_ZONA = float(pd.to_numeric(_mz["itt"], errors="coerce").mean())
# Eventos de seguridad por manzana IPM (join espacial 'within').
def _contar_eventos(nombre):
    _rp = _ruta_itt(nombre)
    if _rp is None:
        return pd.Series(dtype=int)
    _p = gpd.read_file(_rp).to_crs(CRS_WEB)
    _p = _p[_p.geometry.notna()].copy()
    _jo = gpd.sjoin(_p[["geometry"]], ipm_z[["mz_id", "geometry"]], how="inner", predicate="within")
    return _jo.groupby("mz_id").size()
_ch = _contar_eventos("02_seguridad_homicidios.geojson")
_cu = _contar_eventos("03_seguridad_hurtos.geojson")
ipm_z["n_homicidios"] = ipm_z["mz_id"].map(_ch).fillna(0).astype(int)
ipm_z["n_hurtos"] = ipm_z["mz_id"].map(_cu).fillna(0).astype(int)
ipm_z["n_eventos_seg"] = ipm_z["n_homicidios"] + ipm_z["n_hurtos"]
# Umbrales de cuadrante (promedio de la zona).
UMBRAL_IPM = float(ipm_z["ipm"].mean())
UMBRAL_EVE = float(ipm_z["n_eventos_seg"].mean())
IPM_PROM = float(ipm_z["ipm"].mean())
# Variables de privacion (para el perfil de la manzana y el promedio zona).
VAR_LABEL = {
    "ANALF_": "Analfabetismo", "BAJO_": "Bajo logro educativo",
    "INFANCIA_": "Rezago infancia", "INASIS_": "Inasistencia escolar",
    "REZAGO_": "Rezago escolar", "TRAB_INFAN": "Trabajo infantil",
    "DEPEN_": "Dependencia economica", "INFOR_": "Empleo informal",
    "SALUD_": "Barreras salud", "ASEGU_": "Sin aseguramiento",
    "HACI_": "Hacinamiento", "PARED_": "Paredes inadecuadas",
    "EXCRE_": "Sin alcantarillado", "PISOS_": "Pisos inadecuados", "AGUA_": "Sin agua",
}
VARS_PRIV = [v for v in VAR_LABEL if v in ipm_z.columns]
for v in VARS_PRIV:
    ipm_z[v] = pd.to_numeric(ipm_z[v], errors="coerce").fillna(0.0)
PRIV_PROM = {VAR_LABEL[v]: round(float(ipm_z[v].mean()), 1) for v in VARS_PRIV}
print("Manzanas IPM en zona:", len(ipm_z))
print("IPM promedio: %.1f | eventos/manzana promedio: %.1f | ITT (uniforme): %s" % (
    IPM_PROM, UMBRAL_EVE, ("%.1f" % ITT_ZONA if ITT_ZONA is not None else "s/d")))
print("Eventos seguridad: homicidios %d, hurtos %d" % (
    int(ipm_z["n_homicidios"].sum()), int(ipm_z["n_hurtos"].sum())))

## 4. Exportacion de lamina (deck) - helper reutilizable

In [ ]:
# Helper reutilizable para inyectar en un mapa Folium UN boton de exportacion
# de lamina con estructura de deck (PMU). Captura el contenedor del mapa como
# imagen y la compone dentro de una lamina HTML (encabezado + 2 columnas +
# pie) que se exporta a PDF (una pagina). Este archivo es solo una plantilla
# de trabajo; su contenido se inserta como celda en los notebooks.
#
# Decision CORS/tiles: html2canvas no puede rasterizar los tiles base de
# Google/Esri por restricciones CORS (saldrian en blanco o rotos). Por eso,
# antes de capturar el mapa se oculta el panel de tiles base (.leaflet-tile-pane)
# y se pinta un fondo claro (#F4F6F9) en el contenedor; se capturan solo los
# overlays vectoriales (contorno, manzanas, puntos, calor). La imagen del mapa
# va en la columna derecha de la lamina.
def agregar_export(mapa, titulo, slug):
    """Inyecta en 'mapa' (folium.Map) un control flotante con UN boton:
    'Exportar lamina (deck)'. Al pulsar, captura el contenedor del mapa a una
    imagen y construye una lamina HTML con estructura de deck del PMU
    (encabezado dorado, 2 columnas: resumen/cifras clonadas del panel +
    mapa, pie con corte y fuente); captura la lamina y la exporta a PDF de
    una pagina (landscape) con dialogo "Guardar como" (showSaveFilePicker) y
    fallback a descarga. Nombre: deck_<slug>_<AAAA-MM-DD>.pdf. Debe llamarse
    ANTES de mapa.save(...).
    - titulo: titulo visible en el encabezado de la lamina.
    - slug: nombre corto para el archivo.
    """
    map_id = mapa.get_name()  # id del div .folium-map de ESTE mapa
    # Librerias por CDN (sin build).
    cdn = (
        '<script src="https://cdn.jsdelivr.net/npm/html2canvas@1.4.1/dist/html2canvas.min.js"></script>'
        '<script src="https://cdn.jsdelivr.net/npm/jspdf@2.5.1/dist/jspdf.umd.min.js"></script>'
    )
    mapa.get_root().header.add_child(folium.Element(cdn))
    # Control + logica. Se escapa como string de Python; en JS se usa map_id,
    # titulo y slug concretos de ESTE mapa (asi funciona con varios mapas).
    titulo_js = titulo.replace("\\", "\\\\").replace("'", "\\'")
    slug_js = slug.replace("\\", "\\\\").replace("'", "\\'")
    bloque = """
<div id="exportctl___MID__" style="position:fixed;top:12px;right:12px;z-index:10000;
     display:flex;flex-direction:column;gap:6px;font-family:system-ui,Segoe UI,sans-serif">
  <button id="btndeck___MID__" style="cursor:pointer;border:1px solid #333;background:#fff;
     color:#222;padding:6px 10px;border-radius:6px;box-shadow:0 2px 6px rgba(0,0,0,.25);
     font-size:12px;font-weight:600">Exportar lamina (deck)</button>
</div>
<script>
(function(){
  var MAP_ID = '__MID__';
  var TITULO = '__TITULO__';
  var SLUG = '__SLUG__';
  var FUENTE = 'ITT Ciudad Paraiso - CDO Alcaldia de Santiago de Cali';
  var DORADO = '#8A6D1A';
  function fechaISO(){
    var d = new Date();
    var mm = ('0'+(d.getMonth()+1)).slice(-2);
    var dd = ('0'+d.getDate()).slice(-2);
    return d.getFullYear()+'-'+mm+'-'+dd;
  }
  function getContainer(){
    return document.getElementById(MAP_ID);
  }
  // Paso (a): captura SOLO el contenedor del mapa a un canvas, ocultando antes
  // los botones de export, el selector de capas, los tiles base (CORS), el
  // panel flotante de graficas y el slider de intensidad del calor; restaura
  // todo despues. Devuelve el canvas via callback.
  function capturarMapa(cont, cb){
    var ctl = document.getElementById('exportctl_'+MAP_ID);
    if(ctl) ctl.style.display = 'none';
    var layerCtls = cont.querySelectorAll('.leaflet-control-layers');
    layerCtls.forEach(function(lc){ lc.dataset.prevDisplay = lc.style.display; lc.style.display = 'none'; });
    // Panel de graficas flotante (#seg-panel o #eu-panel) y slider del calor.
    var panel = cont.querySelector('#seg-panel, #eu-panel, #itt-stat-panel');
    var prevPanelDisplay = null;
    if(panel){ prevPanelDisplay = panel.style.display; panel.style.display = 'none'; }
    var slider = cont.querySelector('#heat-intensity-ctrl');
    var prevSliderDisplay = null;
    if(slider){ prevSliderDisplay = slider.style.display; slider.style.display = 'none'; }
    // CORS: ocultar tiles base y aplicar fondo claro.
    var tilePanes = cont.querySelectorAll('.leaflet-tile-pane');
    var fondoPrev = cont.style.background;
    cont.style.background = '#F4F6F9';
    tilePanes.forEach(function(tp){ tp.dataset.prevOpacity = tp.style.opacity; tp.style.opacity = '0'; });
    function restaurar(){
      if(ctl) ctl.style.display = '';
      layerCtls.forEach(function(lc){ lc.style.display = ''; });
      if(panel) panel.style.display = (prevPanelDisplay || '');
      if(slider) slider.style.display = (prevSliderDisplay || '');
      tilePanes.forEach(function(tp){ tp.style.opacity = tp.dataset.prevOpacity || ''; });
      cont.style.background = fondoPrev;
    }
    html2canvas(cont, {useCORS:true, backgroundColor:'#F4F6F9', logging:false, scale:2})
      .then(function(canvas){ restaurar(); cb(null, canvas); })
      .catch(function(err){ restaurar(); cb(err, null); });
  }
  // Paso previo: asegura el panel #seg-panel/#eu-panel expandido (si usa
  // <details>) para que Chart.js tenga el canvas visible y pintado, luego
  // captura el canvas real del grafico a un dataURL PNG. Devuelve null si
  // no hay canvas o esta vacio. cb(dataUrl|null).
  function capturarGrafico(cb){
    var panel = document.querySelector('#seg-panel, #eu-panel, #itt-stat-panel');
    if(!panel){ cb(null); return; }
    var dets = panel.querySelectorAll('details');
    dets.forEach(function(d){ d.setAttribute('open', 'open'); });
    // Pequeno tiempo para que el canvas quede visible y Chart.js pinte.
    setTimeout(function(){
      var cv = panel.querySelector('canvas[id*="chart"]') || panel.querySelector('canvas');
      if(!cv || !cv.width || !cv.height){ cb(null); return; }
      try { cb(cv.toDataURL('image/png')); }
      catch(e){ cb(null); }
    }, 250);
  }
  // Paso (b): construye el div-lamina temporal con estructura de deck.
  // La imagen del mapa (canvas de (a)) va en la columna derecha; el panel
  // existente (si lo hay) se clona en la columna izquierda.
  function construirLamina(mapaDataUrl, graficoDataUrl){
    var lam = document.createElement('div');
    lam.id = 'deck-lamina';
    lam.style.cssText = 'position:fixed;top:0;left:0;z-index:2147483647;'+
      'width:1200px;background:#FFFFFF;color:#333;'+
      'font-family:system-ui,Segoe UI,sans-serif;'+
      'box-sizing:border-box;overflow:hidden;';
    // ENCABEZADO (barra con titulo + linea de seccion gris + borde dorado).
    var head = document.createElement('div');
    head.style.cssText = 'padding:16px 22px 12px 22px;border-bottom:3px solid '+DORADO+';';
    head.innerHTML =
      '<div style="font-size:20px;font-weight:700;color:#222;line-height:1.2">'+TITULO+'</div>'+
      '<div style="font-size:12px;color:#888;margin-top:4px">ITT Ciudad Paraiso - Lamina de deck (PMU)</div>';
    lam.appendChild(head);
    // CUERPO en 2 columnas (grid).
    var body = document.createElement('div');
    body.style.cssText = 'display:grid;grid-template-columns:35% 65%;gap:16px;'+
      'padding:16px 22px;align-items:start;';
    // Columna izquierda: resumen / cifras (clon del panel si existe).
    var colIzq = document.createElement('div');
    colIzq.style.cssText = 'min-width:0;';
    var panelOrig = document.querySelector('#seg-panel, #eu-panel, #itt-stat-panel');
    if(panelOrig){
      var clon = panelOrig.cloneNode(true);
      clon.removeAttribute('id');
      // Sacar el clon del flujo fijo para que fluya dentro de la columna.
      clon.style.position = 'static';
      clon.style.top = 'auto'; clon.style.left = 'auto';
      clon.style.right = 'auto'; clon.style.bottom = 'auto';
      clon.style.zIndex = 'auto';
      clon.style.maxWidth = 'none';
      clon.style.width = '100%';
      clon.style.boxShadow = 'none';
      // Expandir cualquier <details> para que la serie se vea.
      var dets = clon.querySelectorAll('details');
      dets.forEach(function(d){ d.setAttribute('open', 'open'); });
      // El canvas de Chart.js se clona VACIO (los canvas no conservan su
      // contenido al clonar). Lo sustituimos por un <img> con la imagen
      // capturada del canvas real para que el grafico SE VEA en la lamina.
      if(graficoDataUrl){
        var cvClon = clon.querySelector('canvas');
        if(cvClon){
          var imgChart = document.createElement('img');
          imgChart.src = graficoDataUrl;
          imgChart.style.cssText = 'display:block;width:100%;height:auto;';
          if(cvClon.parentNode){ cvClon.parentNode.replaceChild(imgChart, cvClon); }
        }
      }
      colIzq.appendChild(clon);
    } else {
      colIzq.innerHTML =
        '<div style="font-size:15px;font-weight:700;color:#222;margin-bottom:6px">'+TITULO+'</div>'+
        '<div style="font-size:13px;color:#666">Resumen no disponible</div>';
    }
    // Columna derecha: imagen del mapa escalada al ancho de la columna.
    var colDer = document.createElement('div');
    colDer.style.cssText = 'min-width:0;';
    var img = document.createElement('img');
    img.src = mapaDataUrl;
    img.style.cssText = 'display:block;width:100%;height:auto;'+
      'border:1px solid #d9dde3;border-radius:6px;';
    colDer.appendChild(img);
    body.appendChild(colIzq);
    body.appendChild(colDer);
    lam.appendChild(body);
    // PIE (corte + fuente + nota).
    var pie = document.createElement('div');
    pie.style.cssText = 'padding:10px 22px 14px 22px;border-top:1px solid #e2e5ea;'+
      'font-size:11px;color:#777;';
    pie.innerHTML = 'Corte: '+new Date().toLocaleString()+
      '  |  Fuente: '+FUENTE+'  |  Mapa ilustrativo';
    lam.appendChild(pie);
    document.body.appendChild(lam);
    return lam;
  }
  // Dialogo "Guardar como" (Chrome/Edge) con fallback a descarga clasica.
  function guardarPDF(blob, nombre){
    function descargaFallback(){
      var url = URL.createObjectURL(blob);
      var a = document.createElement('a');
      a.href = url; a.download = nombre;
      document.body.appendChild(a); a.click(); document.body.removeChild(a);
      setTimeout(function(){ URL.revokeObjectURL(url); }, 1500);
    }
    if(window.showSaveFilePicker){
      var tipos = [{description:'Documento PDF', accept:{'application/pdf':['.pdf']}}];
      window.showSaveFilePicker({suggestedName:nombre, types:tipos})
        .then(function(handle){ return handle.createWritable(); })
        .then(function(writable){ return writable.write(blob).then(function(){ return writable.close(); }); })
        .catch(function(e){
          // Si el usuario cancela (AbortError) no se descarga nada; otros -> fallback.
          if(e && e.name === 'AbortError'){ return; }
          descargaFallback();
        });
    } else {
      descargaFallback();
    }
  }
  // Bounds del poligono de la zona de estudio (Ciudad Paraiso) para
  // reencuadrar el mapa SOLO en la generacion de la lamina (no afecta la
  // vista interactiva previa del usuario; tras exportar queda en este
  // encuadre pero la interactividad se mantiene).
  var ZONA_BOUNDS = L.latLngBounds([[3.442445,-76.53259],[3.44791,-76.526004]]);
  function exportar(){
    var cont = getContainer();
    if(!cont){ alert('No se encontro el contenedor del mapa.'); return; }
    // 1) Forzar el panel a 'open' y capturar el grafico real a dataURL.
    capturarGrafico(function(graficoDataUrl){
      var mp = window[MAP_ID];
      // 2) fitBounds al poligono. maxZoom ENTERO evita el zoom fraccional que
      //    descoloca las capas en html2canvas (desfase al imprimir).
      var capturado = false;
      function trasEncuadre(){
        if(capturado){ return; }
        capturado = true;
        try { if(mp){ mp.invalidateSize(false); } } catch(e){}
        // 3) Margen corto tras asentar panes/tiles, luego capturar.
        setTimeout(function(){
        // 4) Capturar el mapa (ocultando botones/tiles/panel/slider).
        capturarMapa(cont, function(err, canvasMapa){
          if(err){ alert('Error al capturar el mapa: '+err); return; }
          var mapaDataUrl = canvasMapa.toDataURL('image/jpeg', 0.95);
          // 5) Construir lamina (cifras + grafico como imagen + mapa).
          var lam = construirLamina(mapaDataUrl, graficoDataUrl);
          function limpiar(){
            if(lam && lam.parentNode) lam.parentNode.removeChild(lam);
          }
          // 6) Capturar la lamina y generar el PDF.
          html2canvas(lam, {backgroundColor:'#FFFFFF', logging:false, scale:2, useCORS:true})
            .then(function(canvas){
              limpiar();
              var imgData = canvas.toDataURL('image/jpeg', 0.95);
              var jsPDFns = (window.jspdf && window.jspdf.jsPDF) ? window.jspdf.jsPDF : window.jsPDF;
              var w = canvas.width, hh = canvas.height;
              var orient = (w >= hh) ? 'landscape' : 'portrait';
              var doc = new jsPDFns({orientation:orient, unit:'pt', format:[w, hh]});
              doc.addImage(imgData, 'JPEG', 0, 0, w, hh);
              var blob = doc.output('blob');
              var nombre = 'deck_'+SLUG+'_'+fechaISO()+'.pdf';
              guardarPDF(blob, nombre);
            })
            .catch(function(e){
              limpiar();
              alert('Error al exportar la lamina: '+e);
            });
        });
        }, 350);
      }
      // Disparar fitBounds y esperar 'moveend'; fallback por timeout si no llega.
      try {
        if(mp){
          mp.once('moveend', trasEncuadre);
          mp.fitBounds(ZONA_BOUNDS, {padding:[10,10], maxZoom: 17, animate: false});
          setTimeout(trasEncuadre, 1200);
        } else {
          trasEncuadre();
        }
      } catch(e){ trasEncuadre(); }
    });
  }
  function init(){
    var b = document.getElementById('btndeck_'+MAP_ID);
    if(b) b.addEventListener('click', function(){ exportar(); });
  }
  if(document.readyState === 'loading'){
    document.addEventListener('DOMContentLoaded', init);
  } else { init(); }
})();
</script>
"""
    bloque = (bloque.replace("__MID__", map_id)
                    .replace("__TITULO__", titulo_js)
                    .replace("__SLUG__", slug_js))
    mapa.get_root().html.add_child(folium.Element(bloque))
    return mapa

## 5. Construccion del mapa interactivo (Folium)

In [ ]:
# 5a. Mapa de laboratorio ITT x IPM: coropleta IPM clicable + panel analitico reactivo
#      (matriz de 4 cuadrantes + barras comparativas + metrica contextual por capa).
import folium
from folium.plugins import HeatMap
# Rampa IPM (RdPu), coherente con el mapa de IPM dedicado.
RDPU = ["#feebe2", "#fbb4b9", "#f768a1", "#c51b8a", "#7a0177"]
_vmin = float(ipm_z["ipm"].min()); _vmax = float(ipm_z["ipm"].max())
def _col_ipm(v):
    if _vmax <= _vmin:
        return RDPU[2]
    t = (float(v) - _vmin) / (_vmax - _vmin)
    return RDPU[min(int(t * (len(RDPU) - 1) + 0.0001), len(RDPU) - 1)]
# Color de cuadrante (Okabe-Ito) segun IPM y eventos vs umbral.
#   alto IPM + muchos eventos  -> prioridad (bermellon)
#   alto IPM + pocos eventos   -> vulnerable social (naranja)
#   bajo IPM + muchos eventos  -> presion delictiva (azul cielo)
#   bajo IPM + pocos eventos   -> estable (verde)
def _cuadrante(ipm, eve):
    hi_i = ipm >= UMBRAL_IPM
    hi_e = eve >= UMBRAL_EVE
    if hi_i and hi_e: return ("Prioridad (alta vulnerabilidad + alta presion)", "#D55E00")
    if hi_i and not hi_e: return ("Vulnerabilidad social (IPM alto)", "#E69F00")
    if (not hi_i) and hi_e: return ("Presion delictiva (eventos altos)", "#56B4E9")
    return ("Relativamente estable", "#009E73")
if POLIGONO_ZONA is not None:
    _c = POLIGONO_ZONA.geometry.union_all().centroid
    centro = [_c.y, _c.x]
else:
    centro = [ipm_z.geometry.union_all().centroid.y, ipm_z.geometry.union_all().centroid.x]
m = folium.Map(location=centro, zoom_start=16, tiles=None, control_scale=True,
               zoom_snap=0.25, zoom_delta=0.1, max_zoom=22)
m.get_root().width = "100%"; m.get_root().height = "100%"
m.get_root().header.add_child(folium.Element(
    "<style>html,body{height:100%;margin:0;padding:0}"
    ".folium-map{position:absolute;top:0;bottom:0;left:0;right:0;width:100%;height:100%}</style>"))
folium.TileLayer(tiles="https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}",
    attr="Google", name="Google Streets", show=True, max_zoom=22, max_native_zoom=20).add_to(m)
folium.TileLayer(tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri", name="Esri Satelite", show=False, max_zoom=22, max_native_zoom=20).add_to(m)
folium.TileLayer(tiles="https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}",
    attr="Google", name="Google Satelite", show=False, max_zoom=22, max_native_zoom=20).add_to(m)
# Mascara + limite.
if POLIGONO_ZONA is not None:
    _zg = POLIGONO_ZONA.geometry.union_all()
    _mundo = [[-180,-85],[180,-85],[180,85],[-180,85],[-180,-85]]
    _ag = []
    for _g in (list(_zg.geoms) if _zg.geom_type == "MultiPolygon" else [_zg]):
        _ag.append([[x, y] for x, y in _g.exterior.coords])
    folium.GeoJson({"type":"Feature","properties":{},"geometry":{"type":"Polygon","coordinates":[_mundo]+_ag}},
        name="Mascara (foco en la zona)", control=False,
        style_function=lambda f: {"fillColor":"#000","color":"#000","weight":0,"fillOpacity":0.25}).add_to(m)
_rp_lim = _ruta_itt("00_limite_ciudad_paraiso.geojson")
if _rp_lim is not None:
    _lim = gpd.read_file(_rp_lim).to_crs(CRS_WEB)
    _fgl = folium.FeatureGroup(name="Limite Ciudad Paraiso", show=True)
    folium.GeoJson(_lim.to_json(),
        style_function=lambda f: {"color":"#8A6D1A","weight":5,"dashArray":"10 6","fill":False}).add_to(_fgl)
    _fgl.add_to(m)
# Coropleta IPM clicable: cada manzana embebe sus datos para el panel JS.
_fg_ipm = folium.FeatureGroup(name="IPM: pobreza por manzana (" + str(len(ipm_z)) + ")", show=True)
MZ_DATA = []
for _, r in ipm_z.iterrows():
    _col = _col_ipm(r["ipm"])
    _priv = {VAR_LABEL[v]: round(float(r[v]), 1) for v in VARS_PRIV}
    _cu_nom, _cu_col = _cuadrante(float(r["ipm"]), float(r["n_eventos_seg"]))
    _mid = int(r["mz_id"])
    MZ_DATA.append({
        "id": _mid, "ipm": round(float(r["ipm"]), 1),
        "itt": (round(float(r["itt"]), 1) if pd.notna(r.get("itt")) else None),
        "eventos": int(r["n_eventos_seg"]), "homicidios": int(r["n_homicidios"]),
        "hurtos": int(r["n_hurtos"]), "cuadrante": _cu_nom, "color": _cu_col,
        "priv": _priv,
    })
    _gj = folium.GeoJson(
        r["geometry"].__geo_interface__,
        style_function=(lambda f, c=_col: {"fillColor": c, "color": "#7a0177",
                                           "weight": 0.6, "fillOpacity": 0.6}),
        highlight_function=lambda f: {"weight": 2.5, "color": "#111", "fillOpacity": 0.8},
        tooltip="IPM %.1f | eventos %d" % (float(r["ipm"]), int(r["n_eventos_seg"])),
    )
    # Marcar el feature con el id de manzana para capturarlo en el click JS.
    _gj.add_child(folium.Popup("mz:" + str(_mid), show=False))
    _gj.add_to(_fg_ipm)
_fg_ipm.add_to(m)
# Capas de eventos (para probar overlayadd/overlayremove): homicidios y hurtos.
for _arch, _nom, _color in [("02_seguridad_homicidios.geojson", "Eventos: homicidios", "#D55E00"),
                            ("03_seguridad_hurtos.geojson", "Eventos: hurtos", "#0072B2")]:
    _rp = _ruta_itt(_arch)
    if _rp is None:
        continue
    _p = gpd.read_file(_rp).to_crs(CRS_WEB)
    _p = _p[_p.geometry.notna()].copy()
    if POLIGONO_ZONA is not None:
        _p = _p[_p.geometry.within(POLIGONO_ZONA.geometry.union_all())]
    _fg = folium.FeatureGroup(name=_nom + " (" + str(len(_p)) + ")", show=False)
    for _, _r in _p.iterrows():
        folium.CircleMarker([float(_r.geometry.y), float(_r.geometry.x)], radius=3,
            color=_color, weight=1, fill=True, fill_color=_color, fill_opacity=0.7).add_to(_fg)
    _fg.add_to(m)
if POLIGONO_ZONA is not None:
    m.fit_bounds([[3.442445,-76.53259],[3.44791,-76.526004]], padding=(30,30))
folium.LayerControl(collapsed=False, position="topright").add_to(m)
print("Mapa laboratorio construido. Manzanas:", len(MZ_DATA))

## 6. Panel analitico reactivo (matriz + barras + metrica por capa)

In [ ]:
# 6a. Panel analitico reactivo ITT x IPM (matriz 4 cuadrantes + barras + metrica por capa).
_MAP_ID = m.get_name()
# Datos para JS: manzanas, umbrales, promedios y conteos de eventos por manzana.
_eve_por_mz = {int(r["mz_id"]): {"hom": int(r["n_homicidios"]), "hur": int(r["n_hurtos"])}
               for _, r in ipm_z.iterrows()}
_PAYLOAD = json.dumps({
    "manzanas": MZ_DATA,
    "umbral_ipm": round(UMBRAL_IPM, 1),
    "umbral_eve": round(UMBRAL_EVE, 1),
    "ipm_prom": round(IPM_PROM, 1),
    "itt_zona": (round(ITT_ZONA, 1) if ITT_ZONA is not None else None),
    "priv_prom": PRIV_PROM,
    "eve_por_mz": _eve_por_mz,
}, ensure_ascii=False)

_panel = """
<div id="itt-stat-panel" style="position:fixed;top:16px;left:16px;z-index:9999;
     background:__BG__;width:min(430px,42vw);max-height:92vh;overflow:auto;
     padding:12px 14px;border-radius:12px;box-shadow:0 2px 12px rgba(0,0,0,.3);
     font-family:system-ui,Segoe UI,sans-serif;font-size:12px;color:__GRIS__">
  <div style="font-weight:700;font-size:14px;margin-bottom:2px">Laboratorio ITT x IPM</div>
  <div id="lab-ctx" style="font-size:11px;color:#555;margin-bottom:8px;padding:5px 8px;
       background:#eef1f5;border-radius:6px">Clic en una manzana para ver su perfil. Enciende
       una capa de eventos para ver cuantos caen en zonas de IPM alto.</div>

  <div style="font-weight:700;font-size:12px;margin:4px 0">Matriz de priorizacion (IPM vs eventos)</div>
  <div style="font-size:9px;color:#666;margin-bottom:4px">Eje X: IPM (vulnerabilidad). Eje Y: eventos de seguridad por manzana. Lineas = promedio de la zona.</div>
  <div style="position:relative;height:230px;width:100%"><canvas id="lab-scatter"></canvas></div>

  <div id="lab-sel-box" style="display:none;margin-top:8px;border-top:1px solid #e2e5ea;padding-top:8px">
    <div style="font-weight:700;font-size:12px">Manzana seleccionada</div>
    <div id="lab-sel-cuad" style="font-size:11px;margin:2px 0"></div>
    <div style="font-size:9px;color:#666;margin:6px 0 2px">IPM e ITT de la manzana vs promedio de la zona (escala 0-100)</div>
    <div style="position:relative;height:120px;width:100%"><canvas id="lab-bars"></canvas></div>
    <div style="font-size:9px;color:#666;margin:6px 0 2px">Principales privaciones de la manzana (%)</div>
    <div id="lab-priv" style="font-size:11px"></div>
  </div>
</div>
<script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
<script>
(function(){
  var P = __PAYLOAD__;
  var MAP = "__MAP_ID__";
  var scatter=null, bars=null, selId=null, activeEventLayers={};
  function colorPunto(mz){ return mz.color; }
  function buildScatter(){
    var cv=document.getElementById('lab-scatter'); if(!cv||typeof Chart==='undefined'){return;}
    var pts=P.manzanas.map(function(mz){return {x:mz.ipm,y:mz.eventos,mz:mz};});
    scatter=new Chart(cv.getContext('2d'),{type:'scatter',
      data:{datasets:[{label:'Manzanas',data:pts,
        pointBackgroundColor:pts.map(function(p){return colorPunto(p.mz);}),
        pointBorderColor:'#333',pointBorderWidth:0.5,pointRadius:5,pointHoverRadius:7}]},
      options:{responsive:true,maintainAspectRatio:false,
        onClick:function(e,els){ if(els.length){ seleccionar(pts[els[0].index].mz.id,true); } },
        plugins:{legend:{display:false},
          tooltip:{callbacks:{label:function(c){var m=c.raw.mz;return 'IPM '+m.ipm+' | eventos '+m.eventos+' ('+m.cuadrante+')';}}}},
        scales:{x:{title:{display:true,text:'IPM (vulnerabilidad)',font:{size:9}},min:0,ticks:{font:{size:9}}},
                y:{title:{display:true,text:'Eventos de seguridad',font:{size:9}},min:0,ticks:{font:{size:9}}}}}
    });
    dibujarUmbrales();
  }
  // Lineas de umbral (promedio) como anotacion simple via plugin inline.
  function dibujarUmbrales(){
    if(!scatter) return;
    var orig=scatter.draw;
    scatter.draw=function(){ orig.apply(this,arguments);
      var a=this.chartArea, sx=this.scales.x, sy=this.scales.y, ctx=this.ctx;
      ctx.save(); ctx.strokeStyle='#999'; ctx.setLineDash([4,4]); ctx.lineWidth=1;
      var xp=sx.getPixelForValue(P.umbral_ipm); ctx.beginPath(); ctx.moveTo(xp,a.top); ctx.lineTo(xp,a.bottom); ctx.stroke();
      var yp=sy.getPixelForValue(P.umbral_eve); ctx.beginPath(); ctx.moveTo(a.left,yp); ctx.lineTo(a.right,yp); ctx.stroke();
      ctx.restore();
    };
    scatter.update();
  }
  function resaltar(id){
    if(!scatter) return;
    var ds=scatter.data.datasets[0];
    ds.pointRadius=P.manzanas.map(function(mz){return mz.id===id?10:5;});
    ds.pointBorderWidth=P.manzanas.map(function(mz){return mz.id===id?2:0.5;});
    scatter.update();
  }
  function barsManzana(mz){
    var cv=document.getElementById('lab-bars'); if(!cv||typeof Chart==='undefined'){return;}
    var dataMz=[mz.ipm, (mz.itt!==null?mz.itt:0)];
    var dataProm=[P.ipm_prom, (P.itt_zona!==null?P.itt_zona:0)];
    if(bars){ bars.destroy(); }
    bars=new Chart(cv.getContext('2d'),{type:'bar',
      data:{labels:['IPM','ITT'],datasets:[
        {label:'Esta manzana',data:dataMz,backgroundColor:'#CC79A7'},
        {label:'Promedio zona',data:dataProm,backgroundColor:'#999'}]},
      options:{responsive:true,maintainAspectRatio:false,
        plugins:{legend:{labels:{boxWidth:10,font:{size:9}}}},
        scales:{y:{beginAtZero:true,max:100,ticks:{font:{size:9}}},x:{ticks:{font:{size:10}}}}}});
  }
  function seleccionar(id, fromChart){
    selId=id;
    var mz=P.manzanas.find(function(m){return m.id===id;}); if(!mz) return;
    document.getElementById('lab-sel-box').style.display='block';
    document.getElementById('lab-sel-cuad').innerHTML=
      '<span style="display:inline-block;width:11px;height:11px;border-radius:2px;background:'+mz.color+';margin-right:5px"></span>'+
      '<b>'+mz.cuadrante+'</b><br>IPM: <b>'+mz.ipm+'</b> &nbsp; Eventos: <b>'+mz.eventos+'</b> (hom '+mz.homicidios+', hurtos '+mz.hurtos+')'+
      (mz.itt!==null?(' &nbsp; ITT: <b>'+mz.itt+'</b>'):'');
    barsManzana(mz);
    // Top 3 privaciones.
    var arr=Object.keys(mz.priv).map(function(k){return [k,mz.priv[k]];}).sort(function(a,b){return b[1]-a[1];}).slice(0,5);
    document.getElementById('lab-priv').innerHTML=arr.map(function(p){
      return '<div>- '+p[0]+': <b>'+p[1]+'%</b></div>';}).join('');
    resaltar(id);
  }
  // Metrica contextual: cuando se enciende una capa de eventos, cuantos caen en IPM alto.
  function actualizarContexto(){
    var ctx=document.getElementById('lab-ctx');
    var activos=Object.keys(activeEventLayers).filter(function(k){return activeEventLayers[k];});
    if(activos.length===0){
      ctx.innerHTML='Clic en una manzana para ver su perfil. Enciende una capa de eventos para ver cuantos caen en zonas de IPM alto.';
      return;
    }
    // Sumar eventos (del tipo activo) en manzanas con IPM >= umbral.
    var tot=0, enAlto=0;
    P.manzanas.forEach(function(mz){
      var e=P.eve_por_mz[mz.id]||{hom:0,hur:0};
      var n=0;
      if(activeEventLayers['homicidios']) n+=e.hom;
      if(activeEventLayers['hurtos']) n+=e.hur;
      tot+=n; if(mz.ipm>=P.umbral_ipm){ enAlto+=n; }
    });
    var pct = tot>0 ? Math.round(100*enAlto/tot) : 0;
    var nombres=activos.join(' + ');
    ctx.innerHTML='<b>Capa activa:</b> '+nombres+'<br>De <b>'+tot+'</b> eventos, <b>'+enAlto+
      '</b> ('+pct+'%) caen en manzanas de IPM alto (>= '+P.umbral_ipm+').';
  }
  function hookLeaflet(){
    var map=window[MAP];
    if(!map){ setTimeout(hookLeaflet,300); return; }
    // Click en manzana IPM: el popup oculto trae 'mz:<id>'.
    map.eachLayer(function(l){
      if(l&&l.feature&&l.getPopup){ /* geojson */ }
    });
    map.on('popupopen', function(e){
      try{
        var txt=e.popup.getContent();
        if(typeof txt==='string' && txt.indexOf('mz:')===0){
          var id=parseInt(txt.split(':')[1],10);
          e.popup._closeButton && e.popup.remove && e.popup.remove();
          seleccionar(id,false);
        }
      }catch(err){}
    });
    // Deteccion de capas activas por nombre.
    map.on('overlayadd', function(e){
      var n=(e.name||'').toLowerCase();
      if(n.indexOf('homicidios')>=0) activeEventLayers['homicidios']=true;
      if(n.indexOf('hurtos')>=0) activeEventLayers['hurtos']=true;
      actualizarContexto();
    });
    map.on('overlayremove', function(e){
      var n=(e.name||'').toLowerCase();
      if(n.indexOf('homicidios')>=0) delete activeEventLayers['homicidios'];
      if(n.indexOf('hurtos')>=0) delete activeEventLayers['hurtos'];
      actualizarContexto();
    });
  }
  function init(){ buildScatter(); hookLeaflet(); }
  if(document.readyState==='complete'){ setTimeout(init,500); }
  else { window.addEventListener('load',function(){ setTimeout(init,500); }); }
})();
</script>
"""
_panel = (_panel.replace("__BG__", BG).replace("__GRIS__", OKI_GRIS)
                .replace("__PAYLOAD__", _PAYLOAD).replace("__MAP_ID__", _MAP_ID))
m.get_root().html.add_child(folium.Element(_panel))
print("Panel analitico reactivo inyectado (matriz + barras + metrica contextual).")

## 7. Exportacion del HTML

In [ ]:
agregar_export(m, "Laboratorio ITT x IPM", "lab_itt_ipm")
html_lab=os.path.join(MAPAS_DIR,"lab_itt_ipm.html")
m.save(html_lab)
print("Mapa laboratorio exportado:", html_lab)
m